# Final Demo

**Deliverable 04** - Part 3

**Author:** Abdelrahman Yasser Hassan Zaky  
**ID:** 231000102

## Step 1: Load Recommendation Engine

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

with open('../data/recommendation_engine.pkl', 'rb') as f:
    engine = pickle.load(f)

with open('../data/model_comparison_results.pkl', 'rb') as f:
    results = pickle.load(f)

df = engine['df']
print(f'Recommendation engine loaded')
print(f'Dataset: {len(df)} exercises')
print(f'Best model: {max(results, key=lambda m: results[m]["f1"])}')

## Step 2: Interactive Demo Function

In [ ]:
def recommend_exercises(user_level, user_goal, available_equipment, top_n=10):
    filtered = df[df['difficulty'] == user_level].copy()
    filtered = filtered[filtered['equipment'].isin(available_equipment)]
    
    if user_goal == 'cardio':
        goal_types = ['cardio', 'compound']
    elif user_goal == 'strength':
        goal_types = ['push', 'pull', 'isolation', 'legs']
    else:
        goal_types = list(df['type'].unique())
    
    filtered = filtered[filtered['type'].isin(goal_types)]
    
    if len(filtered) > 0:
        return filtered.head(top_n)
    
    fallback = df[df['difficulty'] == user_level].copy()
    fallback = fallback[fallback['type'].isin(goal_types)]
    return fallback.head(top_n)

In [ ]:
def display_recommendations(recs, scenario_name):
    print(f'\n{"=" * 70}')
    print(f'SCENARIO: {scenario_name}')
    print(f'{"=" * 70}')
    
    for i, (_, row) in enumerate(recs.iterrows(), 1):
        print(f'\n  {i}. {row["name"]}')
        print(f'     Target: {row["target_muscles"]}')
        print(f'     Equipment: {row["equipment"]}')
        print(f'     Difficulty: {row["difficulty"]}')
        print(f'     Type: {row["type"]}')
        print(f'     Description: {row["description"]}')

## Step 3: Run Demo Scenarios

In [ ]:
scenario1 = recommend_exercises('beginner', 'cardio', ['bodyweight'])
display_recommendations(scenario1, 'Beginner Cardio (Bodyweight Only)')

In [ ]:
scenario2 = recommend_exercises('intermediate', 'strength', ['barbell', 'dumbbells', 'machine', 'cable machine'])
display_recommendations(scenario2, 'Intermediate Strength (Full Gym)')

In [ ]:
scenario3 = recommend_exercises('advanced', 'hybrid', ['dumbbells', 'bodyweight', 'resistance band'])
display_recommendations(scenario3, 'Advanced Hybrid (Home Gym)')

## Step 4: Summary Statistics

In [ ]:
all_recs = pd.concat([scenario1, scenario2, scenario3])

print('\nTotal recommendations made:', len(all_recs))

print('\nDistribution by Type:')
type_dist = all_recs['type'].value_counts()
for t, c in type_dist.items():
    print(f'  {t}: {c} ({c/len(all_recs)*100:.0f}%)')

print('\nDistribution by Difficulty:')
diff_dist = all_recs['difficulty'].value_counts()
for d, c in diff_dist.items():
    print(f'  {d}: {c} ({c/len(all_recs)*100:.0f}%)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.bar(type_dist.index, type_dist.values, color='coral')
ax1.set_title('Recommendations by Type')
ax1.set_xticklabels(type_dist.index, rotation=45, ha='right')

ax2.bar(diff_dist.index, diff_dist.values, color=['#e74c3c', '#2ecc71', '#f39c12'])
ax2.set_title('Recommendations by Difficulty')

plt.tight_layout()
plt.savefig('../reports/demo_summary.png', dpi=150)
plt.show()